# Lab 20-08 · Delta-Sigma Modulators

**Predict first**, then run. The handout is [README.md](README.md). Levels span [−1, 1]; 0 dBFS is a sine of amplitude 1.

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import dsm

N = 2 ** 16


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {m.group(1): float(m.group(2)) for m in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}


def sine(osr, amp=0.5, phase=0.0):
    m = int(N / (2 * osr) / 3) | 1
    return amp * np.sin(2 * np.pi * m * np.arange(N) / N + phase), m

## Part 1 · Traces and idle tones

*Predict* the integrator values of Examples 20-08.2 and 20-08.7, and where the idle tones of a DC input of 0.002 sit.

In [2]:
print("ideal  :", dsm.mod1(np.full(6, 0.2), u0=0.1)[1][1:])
print("d = 0.5:", dsm.ct1(np.full(6, 0.2), d=0.5, u0=0.1, y_prev=1.0)[1][1:])
r = ngspice("ct_dsm.cir")
print("ngspice:", [r[k] for k in ("u1", "u2", "u3", "u4", "u5")], [r[k] for k in ("d1", "d2", "d3", "d4", "d5")])
for x0 in (0.002, 0.01):
    n = 8000
    y, _ = dsm.mod1(np.full(n, x0), u0=0.1)
    a = 2 * np.abs(np.fft.rfft(y)) / n
    k = round(x0 * n)
    print(f"x = {x0}: line at x fs has amplitude {a[k]:.4f} ({20 * math.log10(a[k]):.1f} dBFS)")

ideal  : [-0.7  0.5 -0.3  0.9  0.1]
d = 0.5: [-0.7 -0.5  0.7  0.9  0.1]


ngspice: [-0.7, 0.499997, -0.3000002, 0.899997, 0.09999983] [-0.7, -0.5000028, 0.6999972, 0.8999992, 0.0999992]
x = 0.002: line at x fs has amplitude 0.0040 (-48.0 dBFS)
x = 0.01: line at x fs has amplitude 0.0200 (-34.0 dBFS)


## Part 2 · Noise shaping

*Predict* the SQNR of first- and second-order loops at OSR = 32, 64 and 128 from the linear model, then simulate.

In [3]:
for osr in (32, 64, 128):
    x, m = sine(osr)
    s1 = dsm.inband_sqnr(dsm.mod1(x)[0], m, osr)
    s2 = dsm.inband_sqnr(dsm.mod2(x)[0], m, osr)
    print(f"OSR {osr:3d}: L=1 {s1:5.1f} dB (model {dsm.sqnr_formula(1, osr, 0.5):5.1f}), "
          f"L=2 {s2:5.1f} dB (model {dsm.sqnr_formula(2, osr, 0.5):5.1f})")
x, m = sine(64)
print("16 levels, L=2, OSR 64:", round(dsm.inband_sqnr(dsm.mod2(x, 16)[0], m, 64), 1), "dB")
f = np.arange(N // 2 + 1) / N
plt.figure()
for yy, lab in ((dsm.mod1(x)[0], "first order"), (dsm.mod2(x)[0], "second order")):
    p = np.abs(np.fft.rfft(yy * np.hanning(N))) ** 2
    plt.semilogx(f[1:], 10 * np.log10(p[1:] / p.max() + 1e-30), lw=0.4, label=lab)
plt.xlabel("f / fs"); plt.ylabel("dB"); plt.legend(); plt.show()

OSR  32: L=1  38.8 dB (model  35.7), L=2  54.3 dB (model  58.1)
OSR  64: L=1  47.7 dB (model  44.8), L=2  70.3 dB (model  73.2)
OSR 128: L=1  54.5 dB (model  53.8), L=2  84.2 dB (model  88.2)


16 levels, L=2, OSR 64: 97.0 dB


## Part 3 · Stability and Lee's rule

*Predict* which third- and fourth-order 1-bit loops stay bounded.

In [4]:
x, m = sine(64)
for order in (3, 4):
    _, wmax = dsm.ef_mod(x, np.poly(np.ones(order)))
    b, a = dsm.lee_ntf(order)
    vals = [dsm.inband_sqnr(dsm.ef_mod(sine(64, phase=k * np.pi / 8)[0], b, a)[0], m, 64) for k in range(8)]
    _, wl = dsm.ef_mod(x, b, a)
    print(f"order {order}: pure NTF max|w| = {wmax:.3g}; Lee NTF max|w| = {wl:.2f}, "
          f"SQNR {np.mean(vals):.1f} dB (spread {min(vals):.1f} to {max(vals):.1f})")

order 3: pure NTF max|w| = inf; Lee NTF max|w| = 1.15, SQNR 80.1 dB (spread 78.7 to 80.8)


order 4: pure NTF max|w| = inf; Lee NTF max|w| = 1.18, SQNR 90.1 dB (spread 89.7 to 90.5)


## Part 4 · MASH and jitter

*Predict* the MASH's SQNR for op-amp gains of 1000 and 100, and the jitter noise of 1-bit and 16-level streams.

In [5]:
x, m = sine(64)
for eps in (0.0, 1e-3, 1e-2):
    print(f"MASH, eps = {eps}: {dsm.inband_sqnr(dsm.mash11(x, eps), m, 64):.1f} dB")
for lev in (2, 16):
    y = dsm.mod2(x, lev)[0]
    pj = dsm.jitter_noise(y, 1e-12, 1.28e9, 32)
    print(f"{lev:2d} levels: E[dy^2] = {np.mean(np.diff(y) ** 2):.4f}, jitter SNR {10 * math.log10(0.125 / pj):.1f} dB")

MASH, eps = 0.0: 73.2 dB
MASH, eps = 0.001: 72.8 dB
MASH, eps = 0.01: 59.7 dB
 2 levels: E[dy^2] = 2.3600, jitter SNR 60.1 dB


16 levels: E[dy^2] = 0.0298, jitter SNR 79.1 dB
